In [1]:
"""
第一周练习（3.15--3.21）
任务1 2 3：
以下函数均满足要求
transfer_vec(vec, old_basis, new_basis)
projection_vec(vec, basis)
angle(vec, basis)
area_scaling_factor(basis)
任务4：
（1）不是所有目标坐标系都可以构成坐标系
（2）构成坐标系，其基向量必须满足之间线性无关（方阵可逆/行列式不等于0）
（3）函数is_valid_basis(basis)实现判断是否构成坐标系的的要求
任务5：
封装成类对象CoordinateSystem
"""
import json  #导入json模块
import numpy as np  #导入numpy包
PATH = 'D:/QG训练营/data(1).json'  #json文件路径

"""--------------------类CoordinateSystem--------------------"""
class CoordinateSystem:
    def __init__(self, cur_basis, vecs=list()):  #参数不为ndarry对象
        self.cur_basis = cur_basis  #初始化类的基向量矩阵（属性）
        self.vecs = vecs  #初始化类的所有向量（属性）

    # 函数--判断基向量是否能够构成坐标系（实现逻辑同下）
    def CS_is_valid_basis(self, basis):
        basis_np = np.array(basis)
        if basis_np.shape[0] != basis_np.shape[1]:  # shape[0]：矩阵的行，代表行基向量个数；shape[1]：矩阵的列，代表基向量的维度
            print(f"传入矩阵{basis_np}不是方阵,返回False")
            return False
        try:
            np.linalg.inv(basis_np)  # 判断方阵是否可逆（方阵可逆等价于基向量线性无关）
            return True
        except np.linalg.LinAlgError:  # LinAlgError（线性代数错误）
            print(f"传入方阵{basis_np}不可逆,不满足构成坐标系的数学要求")
            return False

    # 函数--实现向量转换（实现逻辑基本同下）
    def CS_transfer_vec(self, vec, new_basis):
        norm_vec = np.linalg.norm(vec)  # 检查向量模长
        if norm_vec == 0:
            print(f"向量{vec}模长=0，现返回原vec")
            return vec  # 模长=0，返回原vec
        if is_valid_basis(new_basis):
            print(f"目标坐标系不满足构成坐标系条件，现在返回vec{vec}")
            return vec  # 目标坐标系不满足构成坐标系条件，返回原vec
        old_basis_np = np.array(self.cur_basis, dtype=float)  # 旧基矩阵
        new_basis_np = np.array(new_basis, dtype=float)  # 新基矩阵
        vec_np = np.array(vec, dtype=float)  # 旧向量

        new_vec_np = np.linalg.inv(new_basis_np) @ old_basis_np @ vec_np  # 新向量 = 新基逆矩阵 @ 旧基矩阵 @ 旧向量
        return new_vec_np

    # 函数--计算投影长度（实现逻辑基本同下）
    def CS_projection_vec(self, vec):
        norm_vec = np.linalg.norm(vec)
        if norm_vec == 0:
            print(f"向量{vec}模长=0，现返回原vec")
            return vec  # 模长=0，返回原vec

        vec_np = np.array(vec, dtype=float)
        basis_np = np.array(self.cur_basis, dtype=float)
        arr_proj = list()  # 收集每一个基向量对应的投影长度

        # 逐个基向量求夹角 u：每一个基向量（行向量）
        for u in basis_np:
            norm_u = np.linalg.norm(u)  # 基向量模长
            proj_u = np.dot(vec_np, u) / norm_u  # 向量投影长度 = (基向量 点乘 向量) / 基向量模长
            arr_proj.append(proj_u)

        return np.array(arr_proj)  # 返回ndarry对象（arr_proj列表包含一个向量所有对应基向量的投影）

    # 函数--计算向量与各基向量的夹角（实现逻辑基本同下）
    def CS_angle(self, vec):
        norm_vec = np.linalg.norm(vec)
        if norm_vec == 0:
            print(f"向量{vec}模长=0，现返回原vec")
            return vec  # 模长=0，返回原vec

        vec_np = np.array(vec, dtype=float)
        basis_np = np.array(self.cur_basis, dtype=float)
        arr_angle = list()  # 收集每一个基向量对应的夹角

        for u in basis_np:
            norm_u = np.linalg.norm(u)  # 基向量模长
            norm_vec = np.linalg.norm(vec_np)  # 向量模长
            cos_of_angle = np.dot(vec_np, u) / (norm_u * norm_vec)  # 夹角cos = 两向量点乘 / 两向量模长乘积
            cos_of_angle = np.clip(cos_of_angle, 0, 1)  # 限制cos_of_angle，避免计算精度问题使下面arccos()报错
            real_angle = np.arccos(cos_of_angle)  # arccos()函数，得到真实角度
            arr_angle.append(real_angle)

        return np.array(arr_angle)  # 返回ndarry对象（arr_angle列表包含一个向量与所有对应基向量的夹角）

    # 函数--计算坐标系相对直角坐标系的面积放缩倍数（实现逻辑基本同下）
    def CS_area_scaling_factor(self):
        basis_np = np.array(self.cur_basis, dtype=float)
        return abs(np.linalg.det(basis_np))

"""--------------------全局函数--------------------"""
#函数--加载json文件，返回文件对象
def load_json(path):
    try:  #尝试打开目标文件
        with open(path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"未找到相应文件，路径{path}有误")


#函数--判断基向量是否能够构成坐标系
def is_valid_basis(basis):
    basis_np = np.array(basis)
    if basis_np.shape[0] != basis_np.shape[1]:  #shape[0]：矩阵的行，代表行基向量个数；shape[1]：矩阵的列，代表基向量的维度
        print(f"传入矩阵{basis_np}不是方阵,返回False")
        return False
    try:
        np.linalg.inv(basis_np) #判断方阵是否可逆（方阵可逆等价于基向量线性无关）
        return True
    except np.linalg.LinAlgError:  #LinAlgError（线性代数错误）
        print(f"传入方阵{basis_np}不可逆,不满足构成坐标系的数学要求")
        return False


#函数--实现向量转换
def transfer_vec(vec, old_basis, new_basis):
    norm_vec = np.linalg.norm(vec)  #检查向量模长
    if norm_vec == 0:
        print(f"向量{vec}模长=0，现返回原vec")
        return vec  #模长=0，返回原vec
    if not is_valid_basis(new_basis):
        print(f"目标坐标系不满足构成坐标系条件，现在返回vec{vec}")
        return vec #目标坐标系不满足构成坐标系条件，返回原vec
    old_basis_np = np.array(old_basis, dtype=float)  #旧基矩阵
    new_basis_np = np.array(new_basis, dtype=float)  #新基矩阵
    vec_np = np.array(vec, dtype=float)  #旧向量

    new_vec_np = np.linalg.inv(new_basis_np) @ old_basis_np @ vec_np  #新向量 = 新基逆矩阵 @ 旧基矩阵 @ 旧向量
    return new_vec_np


#函数--计算投影长度
def projection_vec(vec, basis):
    norm_vec = np.linalg.norm(vec)
    if norm_vec == 0:
        print(f"向量{vec}模长=0，现返回原vec")
        return vec  # 模长=0，返回原vec

    vec_np = np.array(vec, dtype=float)
    basis_np = np.array(basis, dtype=float)
    arr_proj = list()  # 收集每一个基向量对应的投影长度

    #逐个基向量求夹角 u：每一个基向量（行向量）
    for u in basis_np:
        norm_u = np.linalg.norm(u)  #基向量模长
        proj_u = np.dot(vec_np, u) / norm_u  #向量投影长度 = (基向量 点乘 向量) / 基向量模长
        arr_proj.append(proj_u)

    return np.array(arr_proj)  #返回ndarry对象（arr_proj列表包含一个向量所有对应基向量的投影）


#函数--计算向量与各基向量的夹角
def angle(vec, basis):
    norm_vec = np.linalg.norm(vec)
    if norm_vec == 0:
        print(f"向量{vec}模长=0，现返回原vec")
        return vec  # 模长=0，返回原vec

    vec_np = np.array(vec, dtype=float)
    basis_np = np.array(basis, dtype=float)
    arr_angle = list()  #收集每一个基向量对应的夹角

    for u in basis_np:
        norm_u = np.linalg.norm(u)  #基向量模长
        norm_vec = np.linalg.norm(vec_np)  #向量模长
        cos_of_angle = np.dot(vec_np, u) / (norm_u * norm_vec)  #夹角cos = 两向量点乘 / 两向量模长乘积
        cos_of_angle = np.clip(cos_of_angle, 0, 1)  #限制cos_of_angle，避免计算精度问题使下面arccos()报错
        real_angle = np.arccos(cos_of_angle)  #arccos()函数，得到真实角度
        arr_angle.append(real_angle)

    return np.array(arr_angle)  #返回ndarry对象（arr_angle列表包含一个向量与所有对应基向量的夹角）


#函数--计算坐标系相对直角坐标系的面积放缩倍数
def area_scaling_factor(basis):
    basis_np = np.array(basis, dtype=float)
    return abs(np.linalg.det(basis_np))



"""--------------------主要流程--------------------"""
if __name__ == "__main__":
    json_data = load_json(PATH)  #读取json文件data(1)
    print("json文件内容及其如下：")
    print(json_data)
    print(f"类型：{type(json_data)}")

    #逐个数据集访问
    for item in json_data:
        print("\n\n\n")
        print(f"\033[31m处理数据组:{item["group_name"]}\033[0m")  #"\033[31m 文本 \033[0m" 使字体显式红色
        cur_old_basis = item["ori_axis"]  #获取这组数据的旧基向量矩阵
        cur_vec = item["vectors"]  #获取这组数据的所有向量
        print(f"cur_vec:{cur_vec}")
        list_tasks = item["tasks"]  #list_tasks是一个字典列表,包含所有待完成任务

        #每个数据组的tasks逐个处理
        for cur_task in list_tasks:
            #处理向量转换
            if cur_task["type"] == "change_axis":
                print("\n============================================================向量转换============================================================\n")
                new_vec = list()  #接收转换好的向量
                cur_new_basis = cur_task["obj_axis"]  #获取这组数据的新基向量矩阵
                #逐个向量转换
                for each_vec in cur_vec:
                    each_vec = transfer_vec(each_vec, cur_old_basis, cur_new_basis)  #转换函数，将旧向量转换成新向量
                    new_vec.append(each_vec)  #将每个更新完毕的vec存入new_vec列表
                cur_vec = new_vec  #更新旧的向量组
                cur_old_basis = cur_new_basis  #更新旧的基向量矩阵
                print(f"转换后的向量{cur_vec}")

            #处理与基向量夹角
            elif cur_task["type"] == "axis_angle":
                print("\n============================================================计算与基向量夹角============================================================\n")
                list_angle = list()  #接收每一个向量与每一个基向量的夹角
                #逐个向量计算其所有夹角
                for each_vec in cur_vec:
                    list_angle.append(angle(each_vec, cur_old_basis))  #函数angle()计算向量与每一个基向量的夹角
                print(f"所有向量与当前基向量的夹角{list_angle}")

            #处理与基向量夹角
            elif cur_task["type"] == "axis_projection":
                print("\n============================================================计算投影长度============================================================\n")
                list_proj = list()  #接收每一个向量与每一个基向量的投影长度
                #逐个向量计算其所有投影长度
                for each_vec in cur_vec:
                    list_proj.append(projection_vec(each_vec, cur_old_basis))  #函数projection_vec()计算向量关于每一个基向量的投影
                print(f"所有向量与当前基向量的投影{list_proj}")

            #处理坐标系面积放缩倍数
            elif cur_task["type"] == "area":
                print("\n============================================================计算面积缩放倍数============================================================\n")
                print(f"当前坐标系面积{area_scaling_factor(cur_old_basis)}")

json文件内容及其如下：
[{'group_name': '2d_task_1', 'vectors': [[1, 3], [1, 2], [2, 4], [3, 1], [4, 3], [5, 5], [6, 2], [7, 7], [8, 6], [9, 8], [10, 9]], 'ori_axis': [[1, 0], [0, 1]], 'tasks': [{'type': 'axis_angle'}, {'type': 'change_axis', 'obj_axis': [[2, 1], [1, 2]]}, {'type': 'area'}, {'type': 'axis_projection'}, {'type': 'axis_angle'}]}, {'group_name': '2d_task_2', 'vectors': [[1, 1], [2, 0], [3, 5], [4, 2], [5, 7], [6, 4], [7, 9], [8, 6], [9, 1], [10, 8], [11, 3], [12, 10]], 'ori_axis': [[1, 1], [1, -1]], 'tasks': [{'type': 'area'}, {'type': 'axis_projection'}, {'type': 'change_axis', 'obj_axis': [[3, 2], [2, -3]]}, {'type': 'axis_projection'}, {'type': 'change_axis', 'obj_axis': [[1, 0], [0, 1]]}, {'type': 'area'}, {'type': 'axis_angle'}]}, {'group_name': '2d_task_3', 'vectors': [[0, 5], [1, 4], [2, 3], [3, 2], [4, 1], [5, 0], [6, -1], [7, -2], [8, -3], [9, -4], [10, -5], [11, -6]], 'ori_axis': [[2, 3], [3, 2]], 'tasks': [{'type': 'axis_projection'}, {'type': 'change_axis', 'obj_axis': 